# SupplyGuard Notebook

This notebook is part of the SupplyGuard final portfolio project.


# 05 Feature Engineering

## Scope

This notebook builds the leakage-safe order-level modeling dataset for the SupplyGuard project.

The goal is to create raw engineered features that can be used later to train machine learning models for late delivery prediction.

The prediction timing is defined as shortly after payment approval. Therefore, features must only use information that would be available at that point or reasonably known before delivery.

This notebook creates the official target variable using the project-wide date-only late delivery definition:

`is_late = delivered_date > estimated_delivery_date`

Orders delivered on the estimated delivery date are not considered late, regardless of delivery timestamp.

This notebook does not perform model preprocessing or modeling. Encoding, scaling, imputation, class imbalance handling, train/test split, model fitting and model evaluation will be handled later in `06_modeling.ipynb` using sklearn/imblearn pipelines and `ColumnTransformer`.

## Expected outputs

- `modeling_dataset.csv`
- `feature_dictionary.csv`
- `feature_engineering_summary.csv`

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

project_root = Path.cwd()
while project_root.name != "supplyguard-delivery-risk" and project_root.parent != project_root:
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"

print(f"Project root: {project_root}")
print(f"Processed data directory: {processed_dir}")

Project root: c:\Users\johan\Desktop\supplyguard-delivery-risk
Processed data directory: c:\Users\johan\Desktop\supplyguard-delivery-risk\data\processed


## Load processed inputs

The modeling dataset is built from the cleaned and aggregated processed tables created in previous notebooks.

The base table is `orders_clean.csv`, enriched with customer, seller, item, product and payment information using the safe order-level join strategy defined during the SQL modeling phase.

Only processed inputs are used in this notebook. Raw data cleaning, relational validation and exploratory delivery analysis were already completed earlier.

In [8]:
input_files = {"orders": "orders_clean.csv", "customers": "customers_clean.csv", "sellers": "sellers_clean.csv",
    "products": "products_clean.csv", "order_items": "order_items_clean.csv",
    "order_items_agg": "order_items_agg.csv", "payments_agg": "payments_agg.csv",
    "geolocation_zip_prefix": "geolocation_zip_prefix_clean.csv"}

dfs = {name: pd.read_csv(processed_dir / file) for name, file in input_files.items()}

orders = dfs["orders"]
customers = dfs["customers"]
sellers = dfs["sellers"]
products = dfs["products"]
order_items = dfs["order_items"]
order_items_agg = dfs["order_items_agg"]
payments_agg = dfs["payments_agg"]
geolocation_zip_prefix = dfs["geolocation_zip_prefix"]

In [9]:
input_overview = pd.DataFrame([{"table": name, "rows": df.shape[0], "columns": df.shape[1], "duplicate_rows": df.duplicated().sum()}
    for name, df in dfs.items()])

input_overview

,table,rows,columns,duplicate_rows
0,orders,99441,8,0
1,customers,99441,5,0
2,sellers,3095,4,0
3,products,32951,10,0
4,order_items,112650,7,0
5,order_items_agg,98666,9,0
6,payments_agg,99440,7,0
7,geolocation_zip_prefix,19015,6,0


## Create official target

The supervised target is created using the project-wide date-only late delivery definition.

A delivered order is classified as late only when the actual customer delivery date is after the estimated delivery date.

The actual delivery timestamp is used only to create the supervised learning label. It will not be included as a predictive feature because it is only known after delivery.

In [10]:
order_date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date"]

orders[order_date_cols] = orders[order_date_cols].apply(pd.to_datetime, errors="coerce")

In [11]:
target_base = orders.copy()

target_base["delivered_date"] = target_base["order_delivered_customer_date"].dt.normalize()
target_base["estimated_delivery_date"] = target_base["order_estimated_delivery_date"].dt.normalize()

valid_target_dates = target_base["delivered_date"].notna() & target_base["estimated_delivery_date"].notna()

target_base["is_late"] = pd.NA
target_base.loc[valid_target_dates, "is_late"] = (target_base.loc[valid_target_dates, "delivered_date"] > target_base.loc[valid_target_dates, "estimated_delivery_date"]
).astype(int)

modeling_base = target_base.loc[(target_base["order_status"] == "delivered") & valid_target_dates].copy()
modeling_base["is_late"] = modeling_base["is_late"].astype(int)

In [12]:
target_summary = pd.DataFrame([
    {"metric": "input_orders", "value": len(orders)},
    {"metric": "orders_with_valid_target_dates", "value": valid_target_dates.sum()},
    {"metric": "delivered_orders_with_valid_target_dates", "value": len(modeling_base)},
    {"metric": "late_orders", "value": modeling_base["is_late"].sum()},
    {"metric": "late_delivery_rate", "value": modeling_base["is_late"].mean()}])

target_summary

,metric,value
0,input_orders,99441.000000
1,orders_with_valid_target_dates,96476.000000
2,delivered_orders_with_valid_target_dates,96470.000000
3,late_orders,6534.000000
4,late_delivery_rate,0.067731


In [13]:
target_distribution = (modeling_base["is_late"].value_counts().rename_axis("is_late").reset_index(name="orders"))

target_distribution["share_pct"] = (target_distribution["orders"] / len(modeling_base) * 100).round(2)

target_distribution

,is_late,orders,share_pct
0,0,89936,93.23
1,1,6534,6.77


In [14]:
pd.DataFrame([
    {"check": "duplicate_order_ids_in_modeling_base", "value": modeling_base["order_id"].duplicated().sum()},
    {"check": "unique_orders_in_modeling_base", "value": modeling_base["order_id"].nunique()}])

,check,value
0,duplicate_order_ids_in_modeling_base,0
1,unique_orders_in_modeling_base,96470


## Prediction timing and leakage policy

The modeling dataset is designed for prediction shortly after payment approval.

At this point, the business can reasonably know information such as customer location, seller location, purchased products, order value, freight value, payment profile, purchase timing and estimated delivery deadline.

Post-delivery information is excluded from the feature set. This includes actual delivery dates, delivery delay calculations, review information and final outcome variables.

The target is created from post-delivery data for supervised learning, but those fields are not used as predictive features.

In [15]:
modeling_features = modeling_base[["order_id", "customer_id", "order_purchase_timestamp", "order_approved_at",
    "order_estimated_delivery_date", "is_late"]].copy()

modeling_features["purchase_month"] = modeling_features["order_purchase_timestamp"].dt.month
modeling_features["purchase_day_of_week"] = modeling_features["order_purchase_timestamp"].dt.dayofweek
modeling_features["purchase_hour"] = modeling_features["order_purchase_timestamp"].dt.hour
modeling_features["is_weekend_purchase"] = modeling_features["purchase_day_of_week"].isin([5, 6]).astype(int)

modeling_features["estimated_delivery_days"] = (modeling_features["order_estimated_delivery_date"].dt.normalize() -modeling_features["order_purchase_timestamp"].dt.normalize()
).dt.days

modeling_features["approval_delay_hours"] = (modeling_features["order_approved_at"] - modeling_features["order_purchase_timestamp"]
).dt.total_seconds() / 3600

In [16]:
customer_features = customers[["customer_id", "customer_zip_code_prefix", "customer_city", "customer_state"]].copy()

modeling_features = modeling_features.merge(customer_features, on="customer_id", how="left")

## Product and seller order-level features

Product and seller information is aggregated to order level before joining the modeling dataset.

This avoids creating multiple rows per order and keeps the final dataset aligned with the supervised learning target.

In [17]:
product_cols = ["product_id", "product_category_name_english", "product_weight_g","product_length_cm", "product_height_cm", "product_width_cm"]

seller_cols = ["seller_id", "seller_state"]

item_context = (order_items.merge(products[product_cols], on="product_id", how="left").merge(sellers[seller_cols], on="seller_id", how="left"))

item_context["product_volume_cm3"] = (item_context["product_length_cm"] * item_context["product_height_cm"] * item_context["product_width_cm"])

In [18]:
product_profile = (
    item_context
    .groupby("order_id").agg(
        product_category_count=("product_category_name_english", "nunique"),
        total_product_weight_g=("product_weight_g", lambda x: x.sum(min_count=1)),
        max_product_weight_g=("product_weight_g", "max"),
        total_product_volume_cm3=("product_volume_cm3", lambda x: x.sum(min_count=1)),
        max_product_volume_cm3=("product_volume_cm3", "max")).reset_index())

dominant_category = (
    item_context
    .groupby(["order_id", "product_category_name_english"], dropna=False).agg(category_item_count=("order_item_id", "count"), category_item_value=("price", "sum"))
    .reset_index().sort_values(["order_id", "category_item_count", "category_item_value"], ascending=[True, False, False])
    .drop_duplicates("order_id")[["order_id", "product_category_name_english"]].rename(columns={"product_category_name_english": "dominant_product_category"}))

product_profile = product_profile.merge(dominant_category, on="order_id", how="left")

In [19]:
main_seller_state = (
    item_context
    .groupby(["order_id", "seller_state"], dropna=False).agg(seller_state_item_count=("order_item_id", "count"), seller_state_item_value=("price", "sum"))
    .reset_index().sort_values(["order_id", "seller_state_item_count", "seller_state_item_value"], ascending=[True, False, False])
    .drop_duplicates("order_id")[["order_id", "seller_state"]].rename(columns={"seller_state": "main_seller_state"}))

seller_profile = (
    order_items_agg[["order_id", "order_item_count", "product_count", "seller_count",
        "total_item_price", "total_freight_value", "avg_item_price", "max_item_price", "total_order_item_value"]]
    .merge(main_seller_state, on="order_id", how="left"))

In [20]:
modeling_features = (
    modeling_features
    .merge(seller_profile, on="order_id", how="left").merge(product_profile, on="order_id", how="left").merge(payments_agg, on="order_id", how="left"))

modeling_features["freight_ratio"] = modeling_features["total_freight_value"] / modeling_features["total_order_item_value"]
modeling_features["same_state_order"] = (modeling_features["customer_state"] == modeling_features["main_seller_state"]).astype("Int64")
modeling_features["cross_state_order"] = (modeling_features["customer_state"] != modeling_features["main_seller_state"]).astype("Int64")
modeling_features["customer_seller_state_pair"] = modeling_features["customer_state"] + "_" + modeling_features["main_seller_state"]

In [21]:
feature_construction_summary = pd.DataFrame([
    {"check": "rows_after_feature_joins", "value": len(modeling_features)},
    {"check": "duplicate_order_ids", "value": modeling_features["order_id"].duplicated().sum()},
    {"check": "columns_after_feature_joins", "value": modeling_features.shape[1]},
    {"check": "late_orders", "value": modeling_features["is_late"].sum()},
    {"check": "late_delivery_rate", "value": modeling_features["is_late"].mean()}
])

feature_construction_summary

,check,value
0,rows_after_feature_joins,96470.000000
1,duplicate_order_ids,0.000000
2,columns_after_feature_joins,40.000000
3,late_orders,6534.000000
4,late_delivery_rate,0.067731


## Geographic distance features

Customer and seller zip code prefixes are enriched with aggregated geolocation coordinates.

These coordinates are used to estimate the approximate customer-seller distance. This feature is available before delivery and can help capture logistics complexity without using post-delivery information.

In [22]:
seller_geography = sellers[["seller_id", "seller_zip_code_prefix", "seller_state"]].copy()

main_seller_geography = (
    order_items
    .merge(seller_geography, on="seller_id", how="left")
    .groupby(["order_id", "seller_zip_code_prefix", "seller_state"], dropna=False).agg(
        seller_geo_item_count=("order_item_id", "count"),
        seller_geo_item_value=("price", "sum")).reset_index()
    .sort_values(["order_id", "seller_geo_item_count", "seller_geo_item_value"], ascending=[True, False, False])
    .drop_duplicates("order_id")[["order_id", "seller_zip_code_prefix"]]
    .rename(columns={"seller_zip_code_prefix": "main_seller_zip_code_prefix"}))

modeling_features = modeling_features.merge(main_seller_geography, on="order_id", how="left")

In [23]:
geo_cols = ["geolocation_zip_code_prefix", "geolocation_lat_median", "geolocation_lng_median"]

customer_geo = geolocation_zip_prefix[geo_cols].rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix", "geolocation_lat_median": "customer_lat", "geolocation_lng_median": "customer_lng"})

seller_geo = geolocation_zip_prefix[geo_cols].rename(columns={
    "geolocation_zip_code_prefix": "main_seller_zip_code_prefix", "geolocation_lat_median": "seller_lat", "geolocation_lng_median": "seller_lng"})

modeling_features = modeling_features.merge(customer_geo, on="customer_zip_code_prefix", how="left").merge(seller_geo, on="main_seller_zip_code_prefix", how="left")

In [24]:
lat1 = np.radians(modeling_features["customer_lat"])
lng1 = np.radians(modeling_features["customer_lng"])
lat2 = np.radians(modeling_features["seller_lat"])
lng2 = np.radians(modeling_features["seller_lng"])

dlat = lat2 - lat1
dlng = lng2 - lng1

a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
modeling_features["customer_seller_distance_km"] = 6371 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

In [25]:
geo_feature_summary = pd.DataFrame([
    {"metric": "rows_after_geo_features", "value": len(modeling_features)},
    {"metric": "duplicate_order_ids", "value": modeling_features["order_id"].duplicated().sum()},
    {"metric": "missing_customer_coordinates", "value": modeling_features["customer_lat"].isna().sum()},
    {"metric": "missing_seller_coordinates", "value": modeling_features["seller_lat"].isna().sum()},
    {"metric": "missing_customer_seller_distance_km", "value": modeling_features["customer_seller_distance_km"].isna().sum()}
])

geo_feature_summary

,metric,value
0,rows_after_geo_features,96470
1,duplicate_order_ids,0
2,missing_customer_coordinates,264
3,missing_seller_coordinates,213
4,missing_customer_seller_distance_km,476


## Final modeling dataset

The final modeling dataset keeps one row per delivered order with a valid target.

Only leakage-safe raw engineered features are retained. High-cardinality identifiers, raw timestamps, actual delivery information and post-delivery outcome fields are excluded.

Missing values are intentionally preserved. Imputation will be handled later inside the modeling pipelines to avoid leakage.

In [26]:
final_columns = [
    "order_id", "is_late",
    "purchase_month", "purchase_day_of_week", "purchase_hour", "is_weekend_purchase", "estimated_delivery_days", "approval_delay_hours",
    "customer_state", "main_seller_state", "same_state_order", "cross_state_order", "customer_seller_state_pair", "customer_seller_distance_km",
    "order_item_count", "product_count", "seller_count", "total_item_price", "total_freight_value", "avg_item_price", "max_item_price", "total_order_item_value", "freight_ratio",
    "product_category_count", "dominant_product_category", "total_product_weight_g", "max_product_weight_g", "total_product_volume_cm3", "max_product_volume_cm3",
    "payment_count", "payment_method_count", "total_payment_value", "avg_payment_value", "max_payment_installments", "main_payment_type"
]

modeling_dataset = modeling_features[final_columns].copy()

In [27]:
final_dataset_summary = pd.DataFrame([
    {"metric": "final_rows", "value": len(modeling_dataset)},
    {"metric": "final_columns", "value": modeling_dataset.shape[1]},
    {"metric": "duplicate_order_ids", "value": modeling_dataset["order_id"].duplicated().sum()},
    {"metric": "late_orders", "value": modeling_dataset["is_late"].sum()},
    {"metric": "late_delivery_rate", "value": modeling_dataset["is_late"].mean()},
    {"metric": "missing_values_total", "value": modeling_dataset.isna().sum().sum()}
])

final_dataset_summary

,metric,value
0,final_rows,96470.000000
1,final_columns,35.000000
2,duplicate_order_ids,0.000000
3,late_orders,6534.000000
4,late_delivery_rate,0.067731
5,missing_values_total,1932.000000


In [28]:
final_target_distribution = modeling_dataset["is_late"].value_counts().rename_axis("is_late").reset_index(name="orders")
final_target_distribution["share_pct"] = (final_target_distribution["orders"] / len(modeling_dataset) * 100).round(2)

final_target_distribution

,is_late,orders,share_pct
0,0,89936,93.23
1,1,6534,6.77


In [29]:
final_missing_summary = (
    modeling_dataset.isna().sum().reset_index(name="missing_values").rename(columns={"index": "column"}))

final_missing_summary["missing_pct"] = (final_missing_summary["missing_values"] / len(modeling_dataset) * 100).round(2)

final_missing_summary.query("missing_values > 0").sort_values("missing_pct", ascending=False)

,column,missing_values,missing_pct
24,dominant_product_category,1372,1.42
13,customer_seller_distance_km,476,0.49
25,total_product_weight_g,16,0.02
26,max_product_weight_g,16,0.02
27,total_product_volume_cm3,16,0.02
28,max_product_volume_cm3,16,0.02
7,approval_delay_hours,14,0.01
29,payment_count,1,0.00
30,payment_method_count,1,0.00
31,total_payment_value,1,0.00


In [30]:
feature_cols = modeling_dataset.drop(columns=["order_id", "is_late"]).columns
numeric_features = modeling_dataset[feature_cols].select_dtypes(include="number").columns.tolist()
categorical_features = [col for col in feature_cols if col not in numeric_features]

feature_type_summary = pd.DataFrame([
    {"feature_type": "numeric", "columns": len(numeric_features)},
    {"feature_type": "categorical", "columns": len(categorical_features)},
    {"feature_type": "target", "columns": 1},
    {"feature_type": "identifier", "columns": 1}
])

feature_type_summary

,feature_type,columns
0,numeric,28
1,categorical,5
2,target,1
3,identifier,1


In [31]:
feature_dictionary = pd.DataFrame([
    {"column": "order_id", "role": "identifier", "description": "Unique order identifier kept for traceability, not used as a predictive feature."},
    {"column": "is_late", "role": "target", "description": "Official date-only late delivery target. 1 if delivered date is after estimated delivery date, else 0."},
    {"column": "purchase_month", "role": "feature", "description": "Month of order purchase."},
    {"column": "purchase_day_of_week", "role": "feature", "description": "Day of week of order purchase, where Monday is 0 and Sunday is 6."},
    {"column": "purchase_hour", "role": "feature", "description": "Hour of day when the order was purchased."},
    {"column": "is_weekend_purchase", "role": "feature", "description": "Flag indicating whether the order was purchased on Saturday or Sunday."},
    {"column": "estimated_delivery_days", "role": "feature", "description": "Calendar days between purchase date and estimated delivery date."},
    {"column": "approval_delay_hours", "role": "feature", "description": "Hours between purchase timestamp and payment approval timestamp."},
    {"column": "customer_state", "role": "feature", "description": "Customer state."},
    {"column": "main_seller_state", "role": "feature", "description": "Main seller state for the order, selected by item count and item value."},
    {"column": "same_state_order", "role": "feature", "description": "Flag indicating whether customer and main seller are in the same state."},
    {"column": "cross_state_order", "role": "feature", "description": "Flag indicating whether customer and main seller are in different states."},
    {"column": "customer_seller_state_pair", "role": "feature", "description": "Combined customer-main seller state pair."},
    {"column": "customer_seller_distance_km", "role": "feature", "description": "Approximate distance in kilometers between customer and main seller zip prefix coordinates."},
    {"column": "order_item_count", "role": "feature", "description": "Number of items in the order."},
    {"column": "product_count", "role": "feature", "description": "Number of unique products in the order."},
    {"column": "seller_count", "role": "feature", "description": "Number of unique sellers in the order."},
    {"column": "total_item_price", "role": "feature", "description": "Total item price before freight."},
    {"column": "total_freight_value", "role": "feature", "description": "Total freight value for the order."},
    {"column": "avg_item_price", "role": "feature", "description": "Average item price in the order."},
    {"column": "max_item_price", "role": "feature", "description": "Maximum item price in the order."},
    {"column": "total_order_item_value", "role": "feature", "description": "Total order item value including item price and freight."},
    {"column": "freight_ratio", "role": "feature", "description": "Freight value divided by total order item value."},
    {"column": "product_category_count", "role": "feature", "description": "Number of unique product categories in the order."},
    {"column": "dominant_product_category", "role": "feature", "description": "Dominant product category selected by category item count and category item value."},
    {"column": "total_product_weight_g", "role": "feature", "description": "Total product weight in grams across order items."},
    {"column": "max_product_weight_g", "role": "feature", "description": "Maximum product weight in grams among order items."},
    {"column": "total_product_volume_cm3", "role": "feature", "description": "Total product volume in cubic centimeters across order items."},
    {"column": "max_product_volume_cm3", "role": "feature", "description": "Maximum product volume in cubic centimeters among order items."},
    {"column": "payment_count", "role": "feature", "description": "Number of payment records associated with the order."},
    {"column": "payment_method_count", "role": "feature", "description": "Number of distinct payment methods used in the order."},
    {"column": "total_payment_value", "role": "feature", "description": "Total payment value associated with the order."},
    {"column": "avg_payment_value", "role": "feature", "description": "Average payment value across payment records."},
    {"column": "max_payment_installments", "role": "feature", "description": "Maximum number of payment installments used in the order."},
    {"column": "main_payment_type", "role": "feature", "description": "Main payment type for the order."}
])

feature_dictionary

,column,role,description
0,order_id,identifier,"Unique order identifier kept for traceability, not used as a predictive feature."
1,is_late,target,"Official date-only late delivery target. 1 if delivered date is after estimated delivery date, else 0."
2,purchase_month,feature,Month of order purchase.
3,purchase_day_of_week,feature,"Day of week of order purchase, where Monday is 0 and Sunday is 6."
4,purchase_hour,feature,Hour of day when the order was purchased.
5,is_weekend_purchase,feature,Flag indicating whether the order was purchased on Saturday or Sunday.
6,estimated_delivery_days,feature,Calendar days between purchase date and estimated delivery date.
7,approval_delay_hours,feature,Hours between purchase timestamp and payment approval timestamp.
8,customer_state,feature,Customer state.
9,main_seller_state,feature,"Main seller state for the order, selected by item count and item value."


In [32]:
feature_engineering_summary = pd.DataFrame([
    {"section": "target", "result": "Created official date-only late delivery target."},
    {"section": "scope", "result": "Restricted modeling dataset to delivered orders with valid actual and estimated delivery dates."},
    {"section": "leakage", "result": "Excluded post-delivery fields, review fields, actual delivery dates and final outcome variables from predictive features."},
    {"section": "temporal_features", "result": "Created purchase timing, weekend, estimated delivery window and approval delay features."},
    {"section": "geography_features", "result": "Created customer/seller state features, state-pair features and approximate customer-seller distance."},
    {"section": "order_economics", "result": "Added item count, product count, seller count, item value, freight value, freight ratio and item price features."},
    {"section": "product_profile", "result": "Added dominant category, category count, product weight and product volume features."},
    {"section": "payment_profile", "result": "Added payment count, payment method count, payment value, installments and main payment type features."},
    {"section": "modeling_preparation", "result": "Kept raw engineered features without encoding, scaling, imputation, class balancing or model training."},
    {"section": "output", "result": "Prepared modeling dataset, feature dictionary and feature engineering summary for downstream modeling."}
])

feature_engineering_summary

,section,result
0,target,Created official date-only late delivery target.
1,scope,Restricted modeling dataset to delivered orders with valid actual and estimated delivery dates.
2,leakage,"Excluded post-delivery fields, review fields, actual delivery dates and final outcome variables from predictive feat..."
3,temporal_features,"Created purchase timing, weekend, estimated delivery window and approval delay features."
4,geography_features,"Created customer/seller state features, state-pair features and approximate customer-seller distance."
5,order_economics,"Added item count, product count, seller count, item value, freight value, freight ratio and item price features."
6,product_profile,"Added dominant category, category count, product weight and product volume features."
7,payment_profile,"Added payment count, payment method count, payment value, installments and main payment type features."
8,modeling_preparation,"Kept raw engineered features without encoding, scaling, imputation, class balancing or model training."
9,output,"Prepared modeling dataset, feature dictionary and feature engineering summary for downstream modeling."


In [33]:
modeling_dataset.to_csv(processed_dir / "modeling_dataset.csv", index=False)
feature_dictionary.to_csv(processed_dir / "feature_dictionary.csv", index=False)
feature_engineering_summary.to_csv(processed_dir / "feature_engineering_summary.csv", index=False)

saved_outputs = pd.DataFrame([
    {"file": "modeling_dataset.csv", "rows": len(modeling_dataset), "columns": modeling_dataset.shape[1]},
    {"file": "feature_dictionary.csv", "rows": len(feature_dictionary), "columns": feature_dictionary.shape[1]},
    {"file": "feature_engineering_summary.csv", "rows": len(feature_engineering_summary), "columns": feature_engineering_summary.shape[1]}
])

saved_outputs

,file,rows,columns
0,modeling_dataset.csv,96470,35
1,feature_dictionary.csv,35,3
2,feature_engineering_summary.csv,10,2


## Final conclusions

This notebook created the final leakage-safe order-level modeling dataset for late delivery prediction.

The official date-only target was successfully created using delivered orders with valid actual and estimated delivery dates. The final dataset contains 96,470 orders, of which 6,534 were late, resulting in a late delivery rate of 6.77%.

The dataset keeps one row per order and includes raw engineered features related to purchase timing, estimated delivery window, customer-seller geography, order economics, product profile and payment profile.

Post-delivery information, review fields, actual delivery dates, delivery delay calculations and final outcome variables were excluded from the predictive feature set.

Missing values were intentionally preserved. No encoding, scaling, imputation, class imbalance handling or model training was performed in this notebook. These steps will be handled later in `06_modeling.ipynb` inside sklearn/imblearn pipelines to avoid leakage and ensure consistent model comparison.

The following outputs were saved to `data/processed/`:

- `modeling_dataset.csv`
- `feature_dictionary.csv`
- `feature_engineering_summary.csv`

The next step is to build the modeling pipeline in `06_modeling.ipynb`, using the saved modeling dataset as the input.